
## File 01 — Centralized Pipeline Logging:


This notebook creates a centralized Delta logging table to track pipeline executions across Bronze, Silver and Gold.

It records:

- Pipeline and task names.
- Run IDs and execution status.
- Start time, end time and duration.
- Processed, successful and failed record counts.
- Error messages and execution details.


## Logging Architecture Flow


```text
+----------------------------------+
| BRONZE / SILVER / GOLD TASKS     |
|                                  |
| Ingestion | Transformation       |
| Validation | MERGE | Analytics   |
+----------------------------------+
                 |
                 v
+----------------------------------+
| INDIVIDUAL PIPELINE EXECUTIONS   |
|                                  |
| Execution ID                     |
| Start Time / End Time            |
| Task Name / Pipeline Name        |
+----------------------------------+
                 |
                 v
+----------------------------------+
| REUSABLE LOGGING FUNCTION        |
|                                  |
| Success / Failure                |
| Duration                         |
| Processing Metrics               |
| Error Details                    |
+----------------------------------+
                 |
                 v
+----------------------------------+
| DELTA LOGGING TABLE              |
|                                  |
| workspace.ecommerce_gold         |
| .pipeline_execution_logs         |
+----------------------------------+
                 |
                 v
+----------------------------------+
| MONITORING AND TROUBLESHOOTING   |
|                                  |
| SQL Queries                      |
| Execution History                |
| Failure Analysis                 |
+----------------------------------+
                 |
                 v
+----------------------------------+
| DASHBOARD REPORTING              |
|                                  |
| Success Rate                     |
| Failed Tasks                     |
| Execution Duration               |
| Processing Metrics               |
+----------------------------------+
```


In [0]:
# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import uuid
import traceback
import logging


# ============================================================
# 2. ENVIRONMENT CONFIGURATION
# ============================================================

CATALOG = "workspace"

BRONZE_SCHEMA = "ecommerce_bronze"
SILVER_SCHEMA = "ecommerce_silver"
GOLD_SCHEMA = "ecommerce_gold"

VOLUME_SCHEMA = "default"
VOLUME = "data_volume"

BASE_PATH = (
    f"/Volumes/{CATALOG}/{VOLUME_SCHEMA}/"
    f"{VOLUME}/ecommerce_lakehouse"
)

LOG_PATH = f"{BASE_PATH}/logs"

LOG_TABLE = (
    f"{CATALOG}.{GOLD_SCHEMA}."
    "pipeline_execution_logs"
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{GOLD_SCHEMA}"
)

dbutils.fs.mkdirs(LOG_PATH)


# ============================================================
# 3. CREATE PYTHON LOGGER
# ============================================================

logger = logging.getLogger(
    "ecommerce_lakehouse_pipeline"
)

logger.setLevel(logging.INFO)

if not logger.handlers:

    console_handler = logging.StreamHandler()

    console_handler.setLevel(logging.INFO)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    )

    console_handler.setFormatter(formatter)

    logger.addHandler(console_handler)


# ============================================================
# 4. DEFINE DELTA LOG TABLE SCHEMA
# ============================================================

log_schema = StructType([

    StructField(
        "log_id",
        StringType(),
        False
    ),

    StructField(
        "run_id",
        StringType(),
        True
    ),

    StructField(
        "pipeline_name",
        StringType(),
        True
    ),

    StructField(
        "task_name",
        StringType(),
        True
    ),

    StructField(
        "dataset_name",
        StringType(),
        True
    ),

    StructField(
        "pipeline_layer",
        StringType(),
        True
    ),

    StructField(
        "execution_status",
        StringType(),
        True
    ),

    StructField(
        "start_timestamp",
        TimestampType(),
        True
    ),

    StructField(
        "end_timestamp",
        TimestampType(),
        True
    ),

    StructField(
        "duration_seconds",
        DoubleType(),
        True
    ),

    StructField(
        "records_read",
        LongType(),
        True
    ),

    StructField(
        "records_processed",
        LongType(),
        True
    ),

    StructField(
        "records_failed",
        LongType(),
        True
    ),

    StructField(
        "error_type",
        StringType(),
        True
    ),

    StructField(
        "error_message",
        StringType(),
        True
    ),

    StructField(
        "execution_details",
        StringType(),
        True
    ),

    StructField(
        "logged_at",
        TimestampType(),
        True
    )

])


# ============================================================
# 5. CREATE LOG TABLE IF NOT EXISTS
# ============================================================

if not spark.catalog.tableExists(LOG_TABLE):

    empty_log_df = spark.createDataFrame(
        [],
        log_schema
    )

    (

        empty_log_df

        .write

        .format("delta")

        .mode("overwrite")

        .saveAsTable(LOG_TABLE)

    )

    logger.info(
        f"Created centralized log table: {LOG_TABLE}"
    )

else:

    logger.info(
        f"Log table already exists: {LOG_TABLE}"
    )


# ============================================================
# 6. GENERATE PIPELINE RUN ID
# ============================================================

def generate_run_id():

    return str(uuid.uuid4())


# ============================================================
# 7. CENTRALIZED LOG WRITER
# ============================================================

def write_pipeline_log(

    run_id,

    pipeline_name,

    task_name,

    dataset_name=None,

    pipeline_layer=None,

    execution_status="SUCCESS",

    start_timestamp=None,

    end_timestamp=None,

    records_read=0,

    records_processed=0,

    records_failed=0,

    error_type=None,

    error_message=None,

    execution_details=None

):

    if start_timestamp is None:

        start_timestamp = datetime.now()

    if end_timestamp is None:

        end_timestamp = datetime.now()

    duration_seconds = (

        end_timestamp - start_timestamp

    ).total_seconds()

    log_record = {

        "log_id": str(uuid.uuid4()),

        "run_id": run_id,

        "pipeline_name": pipeline_name,

        "task_name": task_name,

        "dataset_name": dataset_name,

        "pipeline_layer": pipeline_layer,

        "execution_status": execution_status,

        "start_timestamp": start_timestamp,

        "end_timestamp": end_timestamp,

        "duration_seconds": float(duration_seconds),

        "records_read": int(records_read),

        "records_processed": int(records_processed),

        "records_failed": int(records_failed),

        "error_type": error_type,

        "error_message": (
            str(error_message)[:4000]
            if error_message is not None
            else None
        ),

        "execution_details": (
            str(execution_details)[:4000]
            if execution_details is not None
            else None
        ),

        "logged_at": datetime.now()

    }

    log_df = spark.createDataFrame(
        [log_record],
        schema=log_schema
    )

    (

        log_df

        .write

        .format("delta")

        .mode("append")

        .saveAsTable(LOG_TABLE)

    )

    logger.info(
        f"Log saved: {task_name} | {execution_status}"
    )


# ============================================================
# 8. REUSABLE PIPELINE TASK EXECUTOR
# ============================================================

def execute_pipeline_task(

    run_id,

    pipeline_name,

    task_name,

    pipeline_layer,

    task_function,

    dataset_name=None

):

    start_timestamp = datetime.now()

    logger.info(
        f"Starting task: {task_name}"
    )

    try:

        # Execute the supplied task function.
        # It must return a dictionary of metrics.

        task_result = task_function()

        if task_result is None:

            task_result = {}

        if not isinstance(task_result, dict):

            raise TypeError(
                "Task function must return a dictionary "
                "or None."
            )

        end_timestamp = datetime.now()

        records_read = task_result.get(
            "records_read",
            0
        )

        records_processed = task_result.get(
            "records_processed",
            0
        )

        records_failed = task_result.get(
            "records_failed",
            0
        )

        write_pipeline_log(

            run_id=run_id,

            pipeline_name=pipeline_name,

            task_name=task_name,

            dataset_name=dataset_name,

            pipeline_layer=pipeline_layer,

            execution_status="SUCCESS",

            start_timestamp=start_timestamp,

            end_timestamp=end_timestamp,

            records_read=records_read,

            records_processed=records_processed,

            records_failed=records_failed,

            execution_details=task_result

        )

        logger.info(
            f"Task completed successfully: {task_name}"
        )

        return task_result

    except Exception as error:

        end_timestamp = datetime.now()

        error_trace = traceback.format_exc()

        try:

            write_pipeline_log(

                run_id=run_id,

                pipeline_name=pipeline_name,

                task_name=task_name,

                dataset_name=dataset_name,

                pipeline_layer=pipeline_layer,

                execution_status="FAILED",

                start_timestamp=start_timestamp,

                end_timestamp=end_timestamp,

                error_type=type(error).__name__,

                error_message=str(error),

                execution_details=error_trace

            )

        except Exception as logging_error:

            logger.exception(
                "Failed to write pipeline failure log: "
                f"{logging_error}"
            )

        logger.exception(
            f"Task failed: {task_name}"
        )

        raise


# ============================================================
# 9. CREATE DEMONSTRATION TASK FUNCTIONS
# ============================================================

def bronze_demo_task():

    bronze_table = (
        f"{CATALOG}.{BRONZE_SCHEMA}.customers"
    )

    if not spark.catalog.tableExists(bronze_table):

        raise ValueError(
            f"Bronze table does not exist: {bronze_table}"
        )

    records_read = spark.table(
        bronze_table
    ).count()

    return {

        "records_read": records_read,

        "records_processed": records_read,

        "records_failed": 0,

        "message": "Bronze source table count completed."

    }


def silver_demo_task():

    silver_table = (
        f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
    )

    if not spark.catalog.tableExists(silver_table):

        raise ValueError(
            f"Silver table does not exist: {silver_table}"
        )

    records_read = spark.table(
        silver_table
    ).count()

    return {

        "records_read": records_read,

        "records_processed": records_read,

        "records_failed": 0,

        "message": "Silver customer validation completed."

    }


def gold_demo_task():

    gold_table = (
        f"{CATALOG}.{GOLD_SCHEMA}.gold_daily_sales"
    )

    if not spark.catalog.tableExists(gold_table):

        raise ValueError(
            f"Gold table does not exist: {gold_table}"
        )

    records_read = spark.table(
        gold_table
    ).count()

    return {

        "records_read": records_read,

        "records_processed": records_read,

        "records_failed": 0,

        "message": "Gold daily sales validation completed."

    }


# ============================================================
# 10. EXECUTE DEMONSTRATION PIPELINE
# ============================================================

RUN_ID = generate_run_id()

PIPELINE_NAME = "ecommerce_lakehouse_pipeline"

logger.info(
    f"Pipeline started. Run ID: {RUN_ID}"
)


demo_tasks = [

    {

        "task_name": "bronze_customer_validation",

        "pipeline_layer": "BRONZE",

        "dataset_name": "customers",

        "function": bronze_demo_task

    },

    {

        "task_name": "silver_customer_validation",

        "pipeline_layer": "SILVER",

        "dataset_name": "customers",

        "function": silver_demo_task

    },

    {

        "task_name": "gold_daily_sales_validation",

        "pipeline_layer": "GOLD",

        "dataset_name": None,

        "function": gold_demo_task

    }

]


pipeline_start_time = datetime.now()

successful_tasks = 0

failed_tasks = 0


for task in demo_tasks:

    try:

        execute_pipeline_task(

            run_id=RUN_ID,

            pipeline_name=PIPELINE_NAME,

            task_name=task["task_name"],

            pipeline_layer=task["pipeline_layer"],

            task_function=task["function"],

            dataset_name=task["dataset_name"]

        )

        successful_tasks += 1

    except Exception:

        failed_tasks += 1

        logger.error(
            f"Task failed: {task['task_name']}"
        )

        # Stop the demonstration pipeline on failure.
        break


pipeline_end_time = datetime.now()

pipeline_duration = (
    pipeline_end_time - pipeline_start_time
).total_seconds()


# ============================================================
# 11. WRITE PIPELINE SUMMARY LOG
# ============================================================

write_pipeline_log(

    run_id=RUN_ID,

    pipeline_name=PIPELINE_NAME,

    task_name="pipeline_execution_summary",

    pipeline_layer="PIPELINE",

    execution_status=(
        "SUCCESS"
        if failed_tasks == 0
        else "FAILED"
    ),

    start_timestamp=pipeline_start_time,

    end_timestamp=pipeline_end_time,

    records_read=0,

    records_processed=successful_tasks,

    records_failed=failed_tasks,

    execution_details={

        "successful_tasks": successful_tasks,

        "failed_tasks": failed_tasks,

        "total_tasks": len(demo_tasks),

        "duration_seconds": pipeline_duration

    }

)


# ============================================================
# 12. DISPLAY LATEST PIPELINE LOGS
# ============================================================

print("\n" + "=" * 75)
print("CENTRALIZED PIPELINE LOGS")
print("=" * 75)

latest_logs = (

    spark.table(LOG_TABLE)

    .filter(
        F.col("run_id") == RUN_ID
    )

    .orderBy(
        F.col("logged_at").desc()
    )

)

display(latest_logs)


# ============================================================
# 13. DISPLAY PIPELINE EXECUTION SUMMARY
# ============================================================

print("\nPIPELINE EXECUTION SUMMARY")

print(f"Run ID: {RUN_ID}")

print(f"Successful tasks: {successful_tasks}")

print(f"Failed tasks: {failed_tasks}")

print(
    f"Duration: {pipeline_duration:.2f} seconds"
)

print(
    f"Log table: {LOG_TABLE}"
)


# ============================================================
# 14. FINAL STATUS
# ============================================================

if failed_tasks > 0:

    print("PIPELINE STATUS: FAILED")

    raise RuntimeError(
        "One or more pipeline tasks failed. "
        "Check the centralized log table."
    )

else:

    print("PIPELINE STATUS: SUCCESS")

print(
    "CENTRALIZED LOGGING DEMONSTRATION COMPLETED."
)



### Verify the stored logs

In [0]:
%sql
SELECT
    run_id,
    task_name,
    pipeline_layer,
    execution_status,
    duration_seconds,
    records_read,
    records_processed,
    records_failed,
    error_type,
    error_message,
    logged_at
FROM workspace.ecommerce_gold.pipeline_execution_logs
ORDER BY logged_at DESC;


## Validate the logs using SQL:



### A. View latest execution logs

In [0]:
%sql

SELECT
    run_id,
    pipeline_name,
    task_name,
    pipeline_layer,
    execution_status,
    duration_seconds,
    records_read,
    records_processed,
    records_failed,
    logged_at
FROM workspace.ecommerce_gold.pipeline_execution_logs
ORDER BY logged_at DESC;


## B. Count successful and failed tasks

In [0]:
%sql

SELECT
    pipeline_layer,
    execution_status,
    COUNT(*) AS execution_count
FROM workspace.ecommerce_gold.pipeline_execution_logs
GROUP BY
    pipeline_layer,
    execution_status
ORDER BY
    pipeline_layer,
    execution_status;


### C. Identify failed tasks

In [0]:
%sql

SELECT
    run_id,
    task_name,
    error_type,
    error_message,
    execution_details,
    logged_at
FROM workspace.ecommerce_gold.pipeline_execution_logs
WHERE execution_status = 'FAILED'
ORDER BY logged_at DESC;


### D. Calculate average task execution time

In [0]:
%sql

SELECT
    task_name,
    ROUND(AVG(duration_seconds), 2)
        AS average_duration_seconds,
    MAX(duration_seconds)
        AS maximum_duration_seconds,
    COUNT(*) AS execution_count
FROM workspace.ecommerce_gold.pipeline_execution_logs
WHERE execution_status = 'SUCCESS'
GROUP BY task_name
ORDER BY average_duration_seconds DESC;